# Activity bars with Yahoo OHLCV data

This notebook builds daily activity-based bars using the OHLCV data downloaded from Yahoo Finance.

Generated bar types:

- Time bars
- Count bars as a daily proxy for tick bars
- Volume bars using aggregated universe volume
- Dollar bars using aggregated universe dollar volume


In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "util.py").exists():
    PROJECT_ROOT = next(p for p in PROJECT_ROOT.parents if (p / "util.py").exists())

sys.path.insert(0, str(PROJECT_ROOT / "model" / "preprocessing"))

import pandas as pd

from preprocessing_utils import (
    DATA_OUT,
    PLOTS_DIR,
    build_activity_bars,
    summarize_bars,
    save_bars,
    plot_bar_counts,
    plot_return_distributions,
    plot_bar_durations,
    create_preprocessing_report,
)

TARGET_BARS = int(os.getenv("TARGET_BARS", "1000"))
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_OUT:", DATA_OUT)
print("TARGET_BARS:", TARGET_BARS)


PROJECT_ROOT: /Users/jchulvi/projects/Neural-Networks-Forecasting
DATA_OUT: /Users/jchulvi/projects/Neural-Networks-Forecasting/data/preprocessing
TARGET_BARS: 1000


## Load OHLCV data

In [2]:
ohlcv_path = DATA_OUT / "yahoo_ohlcv.parquet"
if not ohlcv_path.exists():
    raise FileNotFoundError("Run 01_yahoo_ohlcv_audit.ipynb first to create data/preprocessing/yahoo_ohlcv.parquet")

ohlcv = pd.read_parquet(ohlcv_path)
print("OHLCV shape:", ohlcv.shape)
display(ohlcv.head())


OHLCV shape: (16196, 115)


,Close__AEP,Close__BA,Close__CAT,Close__CNP,Close__CVX,Close__DIS,Close__DTE,Close__ED,Close__GD,Close__GE,...,Volume__IP,Volume__JNJ,Volume__KO,Volume__KR,Volume__MMM,Volume__MO,Volume__MRK,Volume__MSI,Volume__PG,Volume__XOM
Date,,,,,,,,,,,,,,,,,,,,,
1962-01-02,0.887649,0.190931,0.462333,0.284116,0.307970,0.057055,0.386574,0.238809,0.168170,0.617245,...,51552,0,806400,153600,254509,345600,633830,65671,192000,902400
1962-01-03,0.886033,0.194749,0.466836,0.281340,0.307275,0.057821,0.383392,0.238809,0.173799,0.611052,...,53736,345600,1574400,131200,505190,1209600,6564672,77611,428800,1200000
1962-01-04,0.873098,0.192840,0.478844,0.281340,0.304494,0.057821,0.380211,0.238072,0.174503,0.603827,...,48494,216000,844800,99200,254509,2592000,1199750,59701,326400,1088000
1962-01-05,0.853696,0.189022,0.483348,0.274553,0.296847,0.058012,0.372256,0.232912,0.175207,0.588344,...,76891,129600,1420800,182400,376979,2937600,520646,107462,544000,1222400
1962-01-08,0.847229,0.189499,0.486350,0.271468,0.295456,0.057821,0.373052,0.234018,0.178021,0.587311,...,93929,172800,2035200,208000,399942,1382400,1380845,89551,1523200,1388800


## Build activity bars

A common bar calendar is built using the aggregated activity of the full asset universe. This keeps the transformed close/return matrices compatible with the multivariate forecasting setup.

In [3]:
bars = build_activity_bars(ohlcv, target_bars=TARGET_BARS)
summary = summarize_bars(bars)

summary_path = DATA_OUT / "activity_bars_summary.csv"
summary.to_csv(summary_path, index=False)

save_bars(bars, DATA_OUT)

print("Saved:", summary_path)
display(summary)


Saved: /Users/jchulvi/projects/Neural-Networks-Forecasting/data/preprocessing/activity_bars_summary.csv


/Users/jchulvi/projects/Neural-Networks-Forecasting/model/preprocessing/preprocessing_utils.py:156: SettingWithCopyWarning: modifications to a property of a datetimelike object are not supported and are discarded. Change values on the original.
  durations.iloc[0] = np.nan
/Users/jchulvi/projects/Neural-Networks-Forecasting/model/preprocessing/preprocessing_utils.py:156: SettingWithCopyWarning: modifications to a property of a datetimelike object are not supported and are discarded. Change values on the original.
  durations.iloc[0] = np.nan
/Users/jchulvi/projects/Neural-Networks-Forecasting/model/preprocessing/preprocessing_utils.py:156: SettingWithCopyWarning: modifications to a property of a datetimelike object are not supported and are discarded. Change values on the original.
  durations.iloc[0] = np.nan
/Users/jchulvi/projects/Neural-Networks-Forecasting/model/preprocessing/preprocessing_utils.py:156: SettingWithCopyWarning: modifications to a property of a datetimelike object a

,bar_type,description,threshold_type,threshold,n_bars,start_date,end_date,mean_days_per_bar,median_days_per_bar,mean_abs_return,return_std,return_skew,return_kurtosis
0,time,Daily time bars,calendar_day,1.000000e+00,16196,1962-01-02,2026-05-08,1.451189,1.0,0.011618,0.017139,-0.396927,19.329904
1,count,Count bars: proxy for tick bars using daily ob...,n_days,1.600000e+01,1013,1962-01-23,2026-05-08,23.202569,23.0,0.048176,0.067380,-0.697980,8.252522
2,volume,Daily volume bars using aggregated universe vo...,total_volume,1.697511e+09,953,1962-07-10,2026-05-08,24.488445,18.0,0.047559,0.070174,-0.391259,12.874495
3,dollar,Daily dollar bars using aggregated universe do...,total_dollar,6.094840e+10,913,1981-01-05,2026-05-08,18.156798,10.0,0.037042,0.062072,1.501154,23.928180


## Generate plots

In [4]:
bar_counts_path = PLOTS_DIR / "activity_bars_counts.png"
return_dist_path = PLOTS_DIR / "activity_bars_return_distributions.png"
durations_path = PLOTS_DIR / "activity_bars_durations.png"

plot_bar_counts(summary, bar_counts_path)
plot_return_distributions(bars, return_dist_path)
plot_bar_durations(bars, durations_path)

print("Saved:", bar_counts_path)
print("Saved:", return_dist_path)
print("Saved:", durations_path)


Saved: /Users/jchulvi/projects/Neural-Networks-Forecasting/data/preprocessing/plots/activity_bars_counts.png
Saved: /Users/jchulvi/projects/Neural-Networks-Forecasting/data/preprocessing/plots/activity_bars_return_distributions.png
Saved: /Users/jchulvi/projects/Neural-Networks-Forecasting/data/preprocessing/plots/activity_bars_durations.png


## Returns distribution summary

In [5]:
return_rows = []
for name, item in bars.items():
    ret = item["returns"]
    per_asset = pd.DataFrame({
        "ticker": ret.columns,
        "bar_type": name,
        "mean_abs_return": ret.abs().mean(axis=0).values,
        "return_std": ret.std(axis=0).values,
        "missing_values": ret.isna().sum(axis=0).values,
    })
    return_rows.append(per_asset)

returns_summary = pd.concat(return_rows, ignore_index=True)
returns_summary_path = DATA_OUT / "returns_distribution_summary.csv"
returns_summary.to_csv(returns_summary_path, index=False)

print("Saved:", returns_summary_path)
display(returns_summary.head(20))


Saved: /Users/jchulvi/projects/Neural-Networks-Forecasting/data/preprocessing/returns_distribution_summary.csv


,ticker,bar_type,mean_abs_return,return_std,missing_values
0,AEP,time,0.008865,0.013078,0
1,BA,time,0.014923,0.021387,0
2,CAT,time,0.013229,0.018673,0
3,CNP,time,0.010683,0.017253,0
4,CVX,time,0.011303,0.016140,0
5,DIS,time,0.013744,0.019823,0
6,DTE,time,0.008518,0.012494,0
7,ED,time,0.008183,0.012591,0
8,GD,time,0.013032,0.018855,0
9,GE,time,0.011942,0.017375,0


## Create report

In [6]:
report_path = create_preprocessing_report(summary)
print("Saved:", report_path)


Saved: /Users/jchulvi/projects/Neural-Networks-Forecasting/model/preprocessing/preprocessing_report.md
